<div class="frontmatter text-center">
<h1> COMP1942: Advanced Analysis: Statistics and Machine Learning</h1>
<h3>Classification models on the Motor Vehicle Insurance dataset</h3>
</div>

# Introduction
This notebook reuses the Week 25 classification workflow (Logistic Regression, KNN, SVC, Naive Bayes, Decision Tree, Random Forest with cross-validation and hyper-parameter tuning) on the **Motor Vehicle Insurance** dataset.

The task: predict whether a policy had **at least one claim during the year** (`Had_Claim`).

## Setup
First, load the required libraries and the dataset.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load the dataset and preview it
dataset = pd.read_excel('Motor vehicle insurance data.xlsx')
dataset.head()

/Applications/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Area,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight
0,1,2015-11-05,2015-11-05,2016-11-05,1956-04-15,1976-03-20,0,4,1,2,...,0,0,2004,80,599,7068.0,0,P,NaN,190
1,1,2015-11-05,2016-11-05,2017-11-05,1956-04-15,1976-03-20,0,4,1,2,...,0,0,2004,80,599,7068.0,0,P,NaN,190
2,1,2015-11-05,2017-11-05,2018-11-05,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.0,0,P,NaN,190
3,1,2015-11-05,2018-11-05,2019-11-05,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.0,0,P,NaN,190
4,2,2017-09-26,2017-09-26,2018-09-26,1956-04-15,1976-03-20,0,4,2,2,...,0,0,2004,80,599,7068.0,0,P,NaN,190


## Data preparation
The raw insurance file is not model-ready, so we tidy it first:
1. Build the target `Had_Claim`.
2. Drop identifier/date columns and claim columns that would leak the target.
3. Convert the `"NA"` text in `Length` / `Type_fuel` to proper missing values, encode the categorical fuel type, and impute.

In [2]:
# 1. Classification target: did the policy have at least one claim this year? (1 = yes, 0 = no)
dataset['Had_Claim'] = (dataset['N_claims_year'] > 0).astype(int)

# 2a. Identifier / raw date columns (not used as features)
id_date_cols = [
    'ID',
    'Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
    'Date_birth', 'Date_driving_licence', 'Date_lapse'
]

# 2b. Claim columns that would leak the target
leakage_cols = ['N_claims_year', 'Cost_claims_year', 'N_claims_history', 'R_Claims_history']

df = dataset.drop(columns=id_date_cols + leakage_cols)

# 3. Make every feature numeric
df['Length']    = pd.to_numeric(df['Length'], errors='coerce')   # 'NA' text -> missing
df['Type_fuel'] = df['Type_fuel'].replace('NA', pd.NA)
df = pd.get_dummies(df, columns=['Type_fuel'], drop_first=True)  # one-hot encode fuel type
df = df.fillna(df.median(numeric_only=True))                     # impute remaining missing values

# Optional: work on a random subsample so GridSearch / SVC stay fast.
# The full dataset has ~105k rows and an RBF SVM is very slow at that size.
# Set USE_SAMPLE = False to train on the entire dataset.
USE_SAMPLE = True
SAMPLE_SIZE = 10000
if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=0)

print('Modelling on', df.shape[0], 'rows and', df.shape[1] - 1, 'features')
df.head()

Modelling on 10000 rows and 19 features


,Distribution_channel,Seniority,Policies_in_force,Max_policies,Max_products,Lapse,Payment,Premium,Type_risk,Area,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Length,Weight,Had_Claim,Type_fuel_P
90160,0,14,1,1,1,0,0,170.18,3,0,0,2004,102,1870,16221.00,5,4.230,1260,0,False
85246,0,9,2,2,1,0,0,288.96,3,0,0,2004,95,1753,19220.00,5,4.174,1241,1,True
94831,0,5,1,1,1,0,0,213.97,2,0,0,2003,60,1769,11088.67,3,3.995,945,0,False
45970,0,9,1,1,1,0,0,289.68,2,0,0,2009,110,1753,16410.00,4,4.308,1581,1,False
41459,0,7,2,2,1,0,0,74.09,3,0,0,2008,100,1390,16295.00,5,4.209,1165,0,True


In [3]:
#  Define the Independent Variables (X) and the Dependent Variable (y)

X = df.drop('Had_Claim', axis=1)
y = df['Had_Claim']

## Relevant functions to avoid unnecessary repetitions and streamline the code

In [4]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=False)
    print(f"\n{model_name} Evaluation")
    print(f"\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nAccuracy Score", accuracy_score(y_test, y_pred))
    print(f"\nClassification Report")
    print(f"\n".join(report.split("\n")[:4]))

In [5]:
def cv_accuracy(model, X_train_used, y_train, model_name, cv_folds=5):
    scores = cross_val_score(model, X_train_used, y_train, cv=cv_folds, scoring="accuracy")
    print(f"{model_name} | CV accuracy mean: {scores.mean():.4f} | std: {scores.std():.4f}")
    return scores

## Splitting the dataset into the Training set and Test set & taking care of necessary scaling

In [6]:
# stratify=y keeps the same claim/no-claim ratio in train and test (the classes are imbalanced)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

ValueError: could not convert string to float: '00/01/1900'

## K-Fold Cross Validation & Hyper-parameter Tuning

# 1. Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=5000)
cv_accuracy(log_reg, X_train_scaled, y_train, "Logistic Regression")

log_reg.fit(X_train_scaled, y_train)
evaluate_model(log_reg, X_test_scaled, y_test, "Logistic Regression")

# 2. K-NN

In [ ]:
knn = KNeighborsClassifier()

param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "p": [1, 2]   # 1=Manhattan, 2=Euclidean
}

grid_knn = GridSearchCV(estimator=knn, param_grid=param_grid_knn, cv=5, scoring="accuracy", n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)
print("\nKNN best params:", grid_knn.best_params_)
print("KNN best CV accuracy:", grid_knn.best_score_)

best_knn = grid_knn.best_estimator_
evaluate_model(best_knn, X_test_scaled, y_test, "KNN (Tuned)")

# 3. SVC

In [ ]:
svc = SVC()

param_grid_svc = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]  # mainly relevant for rbf
}

grid_svc = GridSearchCV(estimator=svc, param_grid=param_grid_svc, cv=5, scoring="accuracy", n_jobs=-1)
grid_svc.fit(X_train_scaled, y_train)
print("\nSVC best params:", grid_svc.best_params_)
print("SVC best CV accuracy:", grid_svc.best_score_)

best_svc = grid_svc.best_estimator_
evaluate_model(best_svc, X_test_scaled, y_test, "SVC (Tuned)")

# 4. Bayes --> do remember, Bayes is praised for its simplicity

In [ ]:
gnb = GaussianNB()
cv_accuracy(gnb, X_train, y_train, "GaussianNB")  # remember: scaling not required

gnb.fit(X_train, y_train)
evaluate_model(gnb, X_test, y_test, "GaussianNB")

# 5. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=0)

param_grid_dt = {
    "max_depth": [None, 3, 5, 10, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4]
}

grid_dt = GridSearchCV(estimator=dt, param_grid=param_grid_dt, cv=5, scoring="accuracy", n_jobs=-1)
grid_dt.fit(X_train, y_train)
print("\nDecision Tree best params:", grid_dt.best_params_)
print("Decision Tree best CV accuracy:", grid_dt.best_score_)

best_dt = grid_dt.best_estimator_
evaluate_model(best_dt, X_test, y_test, "Decision Tree (Tuned)")

# 6. Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=0)

param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=5, scoring="accuracy", n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("\nRandom Forest best params:", grid_rf.best_params_)
print("Random Forest best CV accuracy:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_
evaluate_model(best_rf, X_test, y_test, "Random Forest (Tuned)")

# Review!
### Model performance comparison (accuracy) and CV diagnostic
Fill this in with your own results after running the models above.

| Model | CV accuracy (5-fold) | Test accuracy |
|------|---------------------:|--------------:|
| Random Forest (tuned) |  |  |
| Decision Tree (tuned) |  |  |
| Gaussian Naive Bayes |  |  |
| SVC (tuned) |  |  |
| KNN (tuned) |  |  |
| Logistic Regression |  |  |

> **Note:** claims are the minority class (~19% of policies), so accuracy can look high even for a weak model. It is worth also checking precision/recall/F1 for the claim class (already printed by `evaluate_model`), or re-running with `scoring="f1"` / `"roc_auc"`.